In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Go up two levels: notebooks/evaluation -> notebooks -> project_root
project_root = Path('../..').resolve()
print('Project root:', project_root)

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from uais.data.load_fraud_data import load_fraud_data
from uais.features.fraud_features import build_fraud_feature_table
from uais.supervised.train_fraud_supervised import FraudModelConfig, train_fraud_model
from uais.explainability.shap_explainer import compute_shap_values, plot_shap_summary

from sklearn.model_selection import train_test_split

Project root: /Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2


/Users/pratik_n/Desktop/MyComputer/universal-anomaly-intelligence-v2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_raw = load_fraud_data()
df_feats = build_fraud_feature_table(df_raw, 'Time', 'Amount', 'Class')

target_col = 'Class'
X = df_feats.drop(columns=[target_col])
y = df_feats[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

config = FraudModelConfig(model_type='hist_gb', max_depth=4, learning_rate=0.1, max_iter=200)
model, test_metrics = train_fraud_model(X_train, y_train, X_test, y_test, config)

print('Test metrics:')
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")


Test metrics:
roc_auc: 0.7402
pr_auc: 0.5194
f1: 0.6410
precision: 0.9204
recall: 0.4917
accuracy: 0.9993


In [ ]:
# SHAP Explainability Analysis
import shap
import matplotlib.pyplot as plt

# Extract the actual model from the pipeline
if hasattr(model, 'named_steps'):
    # It's a pipeline - get the 'model' step
    actual_model = model.named_steps.get('model', None)
    if actual_model is None:
        # Try to get the last step
        actual_model = list(model.named_steps.values())[-1]
else:
    actual_model = model

print(f"Model type: {type(actual_model).__name__}")

# Get the transformed features if pipeline has preprocessing
if hasattr(model, 'named_steps') and 'preprocessor' in model.named_steps:
    preprocessor = model.named_steps['preprocessor']
    X_sample = X_test.sample(n=min(500, len(X_test)), random_state=42)
    X_sample_transformed = preprocessor.transform(X_sample)
    feature_names = X_sample.columns.tolist()
else:
    X_sample = X_test.sample(n=min(500, len(X_test)), random_state=42)
    X_sample_transformed = X_sample
    feature_names = X_sample.columns.tolist()

# Create SHAP explainer for the actual model
try:
    explainer = shap.TreeExplainer(actual_model)
    shap_values = explainer.shap_values(X_sample_transformed)
    
    # Handle binary classification case
    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # Get values for positive class
    
    # Create summary plot
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_sample_transformed, feature_names=feature_names[:shap_values.shape[1]], 
                      plot_type="bar", show=False, max_display=15)
    plt.title("SHAP Feature Importance for Fraud Detection", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\n✅ SHAP analysis complete!")
except Exception as e:
    print(f"⚠️ SHAP TreeExplainer failed: {e}")
    print("Using permutation importance instead...")
    
    from sklearn.inspection import permutation_importance
    perm_importance = permutation_importance(model, X_sample, y_test.loc[X_sample.index], n_repeats=10, random_state=42)
    
    # Plot permutation importance
    sorted_idx = perm_importance.importances_mean.argsort()[-15:]
    plt.figure(figsize=(10, 6))
    plt.barh(range(len(sorted_idx)), perm_importance.importances_mean[sorted_idx])
    plt.yticks(range(len(sorted_idx)), [feature_names[i] for i in sorted_idx])
    plt.xlabel("Permutation Importance")
    plt.title("Feature Importance (Permutation-based)", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

InvalidModelError: Model type not yet supported by TreeExplainer: <class 'sklearn.pipeline.Pipeline'>